# Phase 3 - FUNSD relation baseline (Colab runner)

Runner only: mount Drive, pull the Phase 3 branch, fetch the raw FUNSD annotations, run the synthetic unit gate, then run the real FUNSD relation evaluation.

Phase 3 is annotation-only and CPU-only. The FUNSD JSON carries entity text, bbox, label, and GT linking pairs, so this notebook does not load image pixels and does not need a GPU. Logic lives in `src/` and `scripts/`, not in this notebook.

Before running in Colab, make sure `feature/phase3-funsd-relations` has been pushed to GitHub. After Phase 3 merges, set `BRANCH = 'main'` in the boot cell.

## Boot

In [ ]:
# 1. Mount Drive so config.DATA_ROOT and config.OUTPUT_ROOT persist across Colab sessions.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Get the code onto the VM and pin the Phase 3 branch.
import os

REPO = '/content/FinDocStructRAG'
BRANCH = 'feature/phase3-funsd-relations'  # change to 'main' after Phase 3 merges

if not os.path.isdir(f'{REPO}/.git'):
    !git clone --quiet https://github.com/AD2000X/FinDocStructRAG.git {REPO}

!cd {REPO} && git fetch origin --quiet
!cd {REPO} && git checkout {BRANCH} && git pull --ff-only origin {BRANCH}
!cd {REPO} && echo branch: $(git rev-parse --abbrev-ref HEAD) HEAD: $(git log --oneline -1)
%cd /content/FinDocStructRAG

In [ ]:
# 3. Make src/ importable and sanity-check the Phase 3 paths.
import importlib
import sys

sys.path.insert(0, '/content/FinDocStructRAG')
from src import config
importlib.reload(config)

print('IN_COLAB    :', config.IN_COLAB)
print('DATA_ROOT   :', config.DATA_ROOT)
print('OUTPUT_ROOT :', config.OUTPUT_ROOT)
print('FUNSD_ROOT  :', config.FUNSD_ROOT)
print('FUNSD_TRAIN :', config.FUNSD_TRAIN)
print('FUNSD_TEST  :', config.FUNSD_TEST)
print('EVALUATION  :', config.EVALUATION)

## Step 1 - fetch or reuse FUNSD annotations

In [ ]:
# Downloads the official FUNSD zip only if it is not already present on Drive.
# It extracts to data/raw/funsd/dataset/...; tests never depend on this data.
!python scripts/fetch_funsd.py

In [ ]:
# Dataset count sanity check. Expected: 149 train + 50 test annotations.
import importlib
from src import config
importlib.reload(config)

n_train = len(list(config.FUNSD_TRAIN.glob('*.json')))
n_test = len(list(config.FUNSD_TEST.glob('*.json')))
print('train annotations:', n_train)
print('test annotations :', n_test)
assert n_train == 149 and n_test == 50, 'Unexpected FUNSD annotation counts'

## Step 2 - unit gate

In [ ]:
# Small dependency for the synthetic acceptance tests. The Phase 3 runtime itself is stdlib-only.
!python -m pip install -q pytest
!python -m pytest tests/test_funsd_relations.py

## Step 3 - run Phase 3 evaluation

In [ ]:
!python scripts/evaluate_funsd.py

In [ ]:
# Load the JSON report and display the split x scope matrix.
import json
from pathlib import Path

from src import config

report_path = config.EVALUATION / 'phase3_funsd_relations.json'
report = json.loads(report_path.read_text(encoding='utf-8'))
print('report:', report_path)
print('primary:', report['primary'])
print('note   :', report['note'])

rows = []
for split, scopes in report['results'].items():
    for scope, m in scopes.items():
        rows.append({
            'split': split,
            'scope': scope,
            'precision': round(m['precision'], 3),
            'recall': round(m['recall'], 3),
            'f1': round(m['f1'], 3),
            'tp': m['tp'],
            'pred': m['n_pred'],
            'gold': m['n_gold'],
            'forms': m['num_forms'],
        })

try:
    import pandas as pd
    display(pd.DataFrame(rows))
except Exception:
    for row in rows:
        print(row)

## Step 4 - optional error peek

In [ ]:
# Show the lowest-recall held-out forms. This is read-only and does not write artifacts.
from src.funsd_extraction import load_funsd_split, predict_qa_links, qa_gold_links
from src import config

test_forms = load_funsd_split(config.FUNSD_TEST)
miss_rows = []
for form in test_forms:
    pred = predict_qa_links(form)
    gold = qa_gold_links(form)
    tp = len(pred & gold)
    recall = tp / len(gold) if gold else 0.0
    precision = tp / len(pred) if pred else 0.0
    miss_rows.append({
        'form_id': form['form_id'],
        'precision': round(precision, 3),
        'recall': round(recall, 3),
        'tp': tp,
        'pred': len(pred),
        'gold': len(gold),
    })

miss_rows = sorted(miss_rows, key=lambda r: (r['recall'], r['precision'], r['form_id']))[:10]
try:
    import pandas as pd
    display(pd.DataFrame(miss_rows))
except Exception:
    for row in miss_rows:
        print(row)